# M4-Part1: AML CAR-T 临床试验数据库分析

**数据来源**: ClinicalTrials.gov v2 API  
**分析目标**: 梳理 AML CAR-T 主要靶点（CD33 / FLT3 / CD123 / CLEC12A / CD38）的临床试验现状  
**日期**: 2026-05-20  

---

## 研究背景

急性髓系白血病（AML）是成人最常见的急性白血病，预后差，5年生存率约 30%。  
CAR-T 细胞疗法在 ALL 中已获批，但 AML 中面临的挑战更多：
- 靶点在正常造血细胞上也有表达（on-target/off-tumor toxicity）
- AML 表型异质性强，靶点容易丢失（antigen escape）

本分析基于 M1（bulk RNA-seq）和 M2（scRNA-seq）筛选出的候选靶点，
通过临床试验数据验证这些靶点的**临床转化现状**。

**M2 筛选结果（AML vs Normal 倍数）**：
| 靶点 | 倍数 | 等级 |
|------|------|------|
| FLT3 | 10.54x | ⭐⭐⭐ |
| CD33 | 9.92x | ⭐⭐⭐ |
| IL3RA (CD123) | 2.73x | ⭐⭐⭐ |
| CLEC12A | 2.10x | ⭐⭐⭐ |
| CD38 | 1.41x | ⭐⭐ |

## 0. 环境准备

> **⚠️ 注意**: 先运行 `01_clinicaltrials_api.py` 生成 `aml_cart_trials.csv`，再执行本 Notebook

In [ ]:
# 安装依赖（首次运行取消注释）
# !pip install requests pandas matplotlib seaborn

import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
from IPython.display import display

# 图表全局设置
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

PALETTE = ['#4C72B0','#DD8452','#55A868','#C44E52',
           '#8172B2','#937860','#DA8BC3','#8C8C8C']

SCRIPT_DIR = os.path.abspath('.')  # Notebook 所在目录
CSV_PATH   = os.path.join(SCRIPT_DIR, 'aml_cart_trials.csv')

print('✅ 环境加载完毕')
print(f'   数据文件路径: {CSV_PATH}')

## 1. 数据获取

**方法**：调用 ClinicalTrials.gov v2 REST API

```
GET https://clinicaltrials.gov/api/v2/studies
    ?query.term=AML CAR-T CD33
    &pageSize=100
    &format=json
```

对每个靶点独立查询，按 NCTId 去重（一条试验可能同时命中多个靶点关键词）。

In [ ]:
# ─── 如果还没有 CSV，先运行抓取脚本 ───
if not os.path.exists(CSV_PATH):
    print('⚠️  aml_cart_trials.csv 不存在，正在运行抓取脚本...')
    import subprocess
    result = subprocess.run(
        ['python', os.path.join(SCRIPT_DIR, '01_clinicaltrials_api.py')],
        capture_output=True, text=True
    )
    print(result.stdout[-3000:])  # 显示最后 3000 字符
    if result.returncode != 0:
        print('❌ 抓取失败:', result.stderr[-1000:])
else:
    print('✅ 已找到 aml_cart_trials.csv，直接加载')

## 2. 数据加载与清洗

In [ ]:
# 加载 CSV
df = pd.read_csv(CSV_PATH, encoding='utf-8-sig')
print(f'✅ 加载成功: {len(df)} 条临床试验记录')
print(f'   字段数: {df.shape[1]}')
print(f'   列名: {list(df.columns)}')

In [ ]:
# 数据清洗
# 1. 日期列转换
for col in ['StartDate', 'PrimaryCompletionDate', 'CompletionDate']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], format='mixed', errors='coerce')

# 2. 招募人数转数值
df['EnrollmentCount'] = pd.to_numeric(df['EnrollmentCount'], errors='coerce')

# 3. 阶段标准化
phase_map = {
    'PHASE1': 'Phase I', 'PHASE2': 'Phase II', 'PHASE3': 'Phase III',
    'PHASE1, PHASE2': 'Phase I/II', 'PHASE1,PHASE2': 'Phase I/II',
    'PHASE2, PHASE3': 'Phase II/III', 'PHASE2,PHASE3': 'Phase II/III',
    'EARLY_PHASE1': 'Early Phase I', 'N/A': 'N/A',
}
df['PhaseClean'] = df['Phase'].str.strip().str.upper().map(phase_map).fillna(df['Phase'])

# 4. 状态标准化
status_map = {
    'RECRUITING': 'Recruiting',
    'ACTIVE_NOT_RECRUITING': 'Active (Not Recruiting)',
    'COMPLETED': 'Completed',
    'TERMINATED': 'Terminated',
    'WITHDRAWN': 'Withdrawn',
    'SUSPENDED': 'Suspended',
    'NOT_YET_RECRUITING': 'Not Yet Recruiting',
    'ENROLLING_BY_INVITATION': 'Enrolling by Invitation',
}
df['StatusClean'] = df['OverallStatus'].map(status_map).fillna(df['OverallStatus'])

# 5. 拆分靶点列
df['TargetList'] = df['MatchedTargets'].fillna('').str.split('; ')

# 6. 提取启动年份
df['StartYear'] = df['StartDate'].dt.year

print('✅ 数据清洗完成')
display(df[['NCTId','BriefTitle','PhaseClean','StatusClean','EnrollmentCount','StartYear']].head(10))

In [ ]:
# 基本统计
print('=== 数据基本统计 ===')
print(f'\n总记录数: {len(df)}')
print(f'唯一 NCTId: {df["NCTId"].nunique()}')
print(f'\n招募人数统计（有数据的 {df["EnrollmentCount"].notna().sum()} 条）:')
print(df['EnrollmentCount'].describe().round(1))
print(f'\n最早启动年份: {df["StartYear"].min()}')
print(f'最新启动年份: {df["StartYear"].max()}')

## 3. 可视化分析

### 3.1 试验状态分布

In [ ]:
counts = df['StatusClean'].value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# 左：饼图
wedges, texts, autotexts = ax1.pie(
    counts.values,
    labels=counts.index,
    autopct=lambda p: f'{p:.1f}%\n(n={int(round(p*sum(counts.values)/100))})',
    colors=PALETTE[:len(counts)],
    startangle=140,
    pctdistance=0.75,
    wedgeprops=dict(edgecolor='white', linewidth=1.5),
)
for t in autotexts: t.set_fontsize(9)
ax1.set_title('Status Distribution (Pie)', fontweight='bold')

# 右：条形图
bars = ax2.barh(counts.index[::-1], counts.values[::-1],
                color=PALETTE[:len(counts)], edgecolor='white')
for bar, cnt in zip(bars, counts.values[::-1]):
    ax2.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
             str(cnt), va='center', fontsize=10)
ax2.set_xlabel('Number of Studies')
ax2.set_title('Status Distribution (Bar)', fontweight='bold')

fig.suptitle(
    f'AML CAR-T Clinical Trials — Status Overview (n={len(df)})',
    fontsize=14, fontweight='bold', y=1.02
)
fig.tight_layout()
fig.savefig('fig1_status_pie.png', bbox_inches='tight', dpi=150)
plt.show()
print('\n💡 解读：Recruiting 和 Active 的比例反映了该领域的活跃程度')

### 3.2 试验阶段分布

In [ ]:
phase_order = ['Early Phase I','Phase I','Phase I/II','Phase II','Phase II/III','Phase III','Phase IV','N/A']
phase_order = [p for p in phase_order if p in df['PhaseClean'].values]

pivot = (
    df.groupby(['PhaseClean','StatusClean'])
      .size().unstack(fill_value=0)
      .reindex(phase_order, fill_value=0)
)

fig, ax = plt.subplots(figsize=(11, 6))
pivot.plot(kind='bar', ax=ax, stacked=True,
           color=PALETTE[:pivot.shape[1]], edgecolor='white', linewidth=0.5)

ax.set_xlabel('Clinical Trial Phase', fontsize=12)
ax.set_ylabel('Number of Studies', fontsize=12)
ax.set_title(
    f'AML CAR-T Clinical Trials — Phase Distribution (n={len(df)})',
    fontsize=13, fontweight='bold'
)
ax.legend(title='Status', bbox_to_anchor=(1.02,1), loc='upper left', fontsize=9)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))

fig.tight_layout()
fig.savefig('fig2_phase_bar.png', bbox_inches='tight', dpi=150)
plt.show()
print('\n💡 解读：AML CAR-T 大多处于 Phase I/II，说明该领域仍处早期探索阶段')

### 3.3 各靶点试验数量

In [ ]:
# 拆分靶点，统计命中次数
target_counter = Counter()
for targets in df['TargetList']:
    for t in targets:
        if t and t not in ('Unknown', '', 'nan'):
            target_counter[t] += 1

if target_counter:
    targets_sorted = sorted(target_counter, key=lambda x: -target_counter[x])
    counts_t = [target_counter[t] for t in targets_sorted]

    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(targets_sorted, counts_t,
                  color=PALETTE[:len(targets_sorted)], edgecolor='white')
    for bar, cnt in zip(bars, counts_t):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                str(cnt), ha='center', va='bottom', fontsize=11, fontweight='bold')

    ax.set_xlabel('Target', fontsize=12)
    ax.set_ylabel('Number of Studies', fontsize=12)
    ax.set_title(
        'AML CAR-T Clinical Trials — by Target\n(one study may match multiple targets)',
        fontsize=13, fontweight='bold'
    )
    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    fig.tight_layout()
    fig.savefig('fig3_target_bar.png', bbox_inches='tight', dpi=150)
    plt.show()
    print('\n💡 解读：CD33 是 AML CAR-T 领域最成熟的靶点（Gemtuzumab 已批），FLT3 紧随其后')
else:
    print('⚠️  没有靶点数据')

### 3.4 试验启动时间线

In [ ]:
df_dated = df.dropna(subset=['StartYear']).copy()
df_dated['StartYear'] = df_dated['StartYear'].astype(int)
df_dated = df_dated[df_dated['StartYear'] >= 2010]

year_status = (
    df_dated.groupby(['StartYear','StatusClean'])
            .size().unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(13, 6))
year_status.plot(kind='bar', ax=ax, stacked=True,
                 color=PALETTE[:year_status.shape[1]], edgecolor='white', linewidth=0.3)

ax.set_xlabel('Start Year', fontsize=12)
ax.set_ylabel('Number of Studies', fontsize=12)
ax.set_title(
    'AML CAR-T Clinical Trials — Launch Timeline (2010–present)',
    fontsize=13, fontweight='bold'
)
ax.legend(title='Status', bbox_to_anchor=(1.02,1), loc='upper left', fontsize=9)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))

fig.tight_layout()
fig.savefig('fig4_timeline.png', bbox_inches='tight', dpi=150)
plt.show()
print('\n💡 解读：2018 年后 AML CAR-T 临床试验数量显著增加，呈现明显的时间加速趋势')

### 3.5 国家分布 Top 10

In [ ]:
country_counter = Counter()
for cstr in df['Countries'].dropna():
    for c in str(cstr).split('; '):
        c = c.strip()
        if c:
            country_counter[c] += 1

if country_counter:
    top10 = country_counter.most_common(10)
    labels = [x[0] for x in top10]
    counts_c = [x[1] for x in top10]

    fig, ax = plt.subplots(figsize=(9, 6))
    bars = ax.barh(labels[::-1], counts_c[::-1],
                   color=PALETTE[0], edgecolor='white')
    for bar, cnt in zip(bars, counts_c[::-1]):
        ax.text(bar.get_width()+0.2, bar.get_y()+bar.get_height()/2,
                str(cnt), va='center', fontsize=10)

    ax.set_xlabel('Number of Studies', fontsize=12)
    ax.set_title('AML CAR-T Clinical Trials — Top 10 Countries',
                 fontsize=13, fontweight='bold')
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    fig.tight_layout()
    fig.savefig('fig5_country_top10.png', bbox_inches='tight', dpi=150)
    plt.show()
    print('\n💡 解读：美国和中国是 AML CAR-T 临床试验最活跃的国家，中国贡献大量 Phase I')

## 4. 靶点 × 阶段透视表

In [ ]:
# 展开靶点列（一行变多行）
exploded = df.explode('TargetList')
exploded = exploded[
    exploded['TargetList'].notna() &
    (exploded['TargetList'] != '') &
    (exploded['TargetList'] != 'Unknown')
]

if len(exploded) > 0:
    pivot = (
        exploded.groupby(['TargetList','PhaseClean'])
                .size().unstack(fill_value=0)
    )
    pivot['Total'] = pivot.sum(axis=1)
    pivot = pivot.sort_values('Total', ascending=False)

    print('📋 靶点 × 阶段透视表：')
    display(pivot)

    # 保存
    pivot.to_csv('summary_table.csv', encoding='utf-8-sig')
    print('\n✅ 已保存: summary_table.csv')
else:
    print('⚠️  没有靶点数据')

## 5. 关键发现汇总

In [ ]:
print('=' * 60)
print('  M4-Part1 关键发现汇总')
print('=' * 60)

print(f'\n📊 总体规模:')
print(f'   AML CAR-T 相关临床试验: {len(df)} 条')
print(f'   最早试验启动年份: {int(df["StartYear"].min())} 年')
print(f'   中位招募人数: {df["EnrollmentCount"].median():.0f} 人/项目')

print(f'\n📊 试验阶段:')
phase_counts = df['PhaseClean'].value_counts()
for phase, cnt in phase_counts.items():
    print(f'   {phase:<20s}: {cnt}')

print(f'\n📊 与 M2 scRNA-seq 候选靶点的对应关系:')
target_hits = {
    'CD33':    target_counter.get('CD33', 0),
    'FLT3':    target_counter.get('FLT3', 0),
    'CD123':   target_counter.get('CD123', 0),
    'CLEC12A': target_counter.get('CLEC12A', 0) + target_counter.get('CLL-1', 0),
    'CD38':    target_counter.get('CD38', 0),
}
print(f'   {"靶点":<12} {"M2倍数":<12} {"临床试验数"}')
print(f'   {"-"*36}')
m2_fold = {'CD33': '9.92x', 'FLT3': '10.54x', 'CD123': '2.73x', 'CLEC12A': '2.10x', 'CD38': '1.41x'}
for t, hits in sorted(target_hits.items(), key=lambda x: -x[1]):
    print(f'   {t:<12} {m2_fold.get(t, "N/A"):<12} {hits}')

print('\n✅ Part1 分析完成！下一步：M4-Part2 孟德尔随机化分析')

---

## 结论

通过 ClinicalTrials.gov API 系统梳理了 AML CAR-T 领域的临床试验现状：

1. **CD33 和 FLT3 是临床转化最活跃的靶点**，与 M2 scRNA-seq 分析中的高表达倍数一致
2. **该领域以 Phase I/II 为主**，整体仍处于早期临床探索阶段，空间巨大
3. **2018 年后试验数量显著增加**，尤其是中国贡献了大量 Phase I 试验
4. **CLEC12A (CLL-1)** 虽然在 M1 中是最显著上调靶点（logFC=2.78），
   但临床试验数量相对少，说明这是一个**尚未充分开发的新兴靶点**，研究价值高

**对 Portfolio 的意义**：本分析将组学数据（M1/M2）与临床转化进展（M4-Part1）
联系起来，形成从「发现靶点」到「验证临床价值」的完整研究逻辑链。

---
*数据来源: ClinicalTrials.gov | 分析日期: 2026-05-20*